<a href="https://colab.research.google.com/github/Godstouch/GNN-Student-Risk-Prediction-/blob/main/Gated_GAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
if 'google.colab' in sys.modules:
  !pip install -q torch_geometric

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
from torch_geometric.nn import GATConv
from sklearn.metrics import f1_score, classification_report, recall_score

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
data = torch.load('/content/real_graph_relabeled.pt', weights_only=False).to(device)
num_classes = int(data.y.max().item()) + 1
num_features = data.x.shape[1]


class GatedGAT(nn.Module):

    def __init__(self, in_dim, hidden_dim, out_dim, heads=8, dropout=0.5):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden_dim, heads=heads, dropout=dropout)
        self.gat2 = GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False, dropout=dropout)
        self.skip = nn.Linear(in_dim, hidden_dim)
        self.gate = nn.Linear(hidden_dim * 2, hidden_dim)  # learned gate, one value per hidden dim
        self.out = nn.Linear(hidden_dim, out_dim)
        self.dropout = dropout

    def forward(self, x, edge_index):
        h = F.dropout(x, p=self.dropout, training=self.training)
        h = F.elu(self.gat1(h, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = self.gat2(h, edge_index)

        s = F.relu(self.skip(x))

        g = torch.sigmoid(self.gate(torch.cat([h, s], dim=-1)))
        z = g * h + (1 - g) * s
        z = F.dropout(z, p=self.dropout, training=self.training)
        return self.out(z), g


def train_gnn(model, data, epochs=300, lr=0.01, weight_decay=5e-4, patience=40, verbose=True, class_weights=None):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    best_val_f1 = -1; best_state = None; patience_ctr = 0
    history = {'train_loss': [], 'val_f1': []}
    for epoch in range(1, epochs + 1):
        model.train(); optimizer.zero_grad()
        out, _ = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask], weight=class_weights)
        loss.backward(); optimizer.step()

        model.eval()
        with torch.no_grad():
            out, _ = model(data.x, data.edge_index)
            pred = out[data.val_mask].argmax(dim=1).cpu().numpy()
            true = data.y[data.val_mask].cpu().numpy()
            val_f1 = f1_score(true, pred, average='macro')

        history['train_loss'].append(loss.item()); history['val_f1'].append(val_f1)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1

        if verbose and epoch % 20 == 0:
            print(f"Epoch {epoch}: loss={loss.item():.4f}, val_macro_f1={val_f1:.4f}")
        if patience_ctr >= patience:
            if verbose: print(f"Early stopping at epoch {epoch}, best val macro-F1={best_val_f1:.4f}")
            break
    model.load_state_dict(best_state)
    return history


def evaluate(model, data, mask):
    model.eval()
    with torch.no_grad():
        out, gate = model(data.x, data.edge_index)
        pred = out[mask].argmax(dim=1).cpu().numpy()
        true = data.y[mask].cpu().numpy()
        mean_gate = gate[mask].mean().item()
    macro_f1 = f1_score(true, pred, average='macro')
    per_class = f1_score(true, pred, average=None)
    per_class_f1 = dict(zip(['High-Risk', 'Moderate-Risk', 'Low-Risk'], per_class))
    high_risk_recall = recall_score(true, pred, labels=[0], average='macro')
    return {'macro_f1': macro_f1, 'per_class_f1': per_class_f1, 'high_risk_recall': high_risk_recall,
            'mean_gate': mean_gate, 'report': classification_report(true, pred, digits=3)}


set_seed(42)
counts = torch.bincount(data.y[data.train_mask])
class_weights = (counts.sum() / (num_classes * counts.float())).to(device)
print("class_weights:", class_weights.tolist())

model_gated_gat = GatedGAT(num_features, hidden_dim=32, out_dim=num_classes, heads=8, dropout=0.5).to(device)
print("Training GatedGAT model...")
history = train_gnn(model_gated_gat, data, epochs=300, lr=0.01, weight_decay=5e-4, patience=40,
                     verbose=True, class_weights=class_weights)
print("Training complete.")

eval_results = evaluate(model_gated_gat, data, data.test_mask)
print(f"\nMacro-F1 (GatedGAT): {eval_results['macro_f1']:.4f}")
print("Per-class F1:")
for cn, f1 in eval_results['per_class_f1'].items():
    print(f"  {cn}: {f1:.4f}")
print(f"High-Risk Recall (GatedGAT): {eval_results['high_risk_recall']:.4f}")
print(f"Mean learned gate value on test set (0=trust own features, 1=trust neighbors): {eval_results['mean_gate']:.4f}")
print(eval_results['report'])

class_weights: [1.0144927501678467, 0.9887005686759949, 0.9971510171890259]
Training GatedGAT model...
Epoch 20: loss=0.5225, val_macro_f1=0.7831
Epoch 40: loss=0.2827, val_macro_f1=0.8002
Epoch 60: loss=0.2178, val_macro_f1=0.8214
Epoch 80: loss=0.1516, val_macro_f1=0.8007
Early stopping at epoch 88, best val macro-F1=0.8352
Training complete.

Macro-F1 (GatedGAT): 0.8545
Per-class F1:
  High-Risk: 0.9796
  Moderate-Risk: 0.7961
  Low-Risk: 0.7879
High-Risk Recall (GatedGAT): 0.9796
Mean learned gate value on test set (0=trust own features, 1=trust neighbors): 0.1763
              precision    recall  f1-score   support

           0      0.980     0.980     0.980        49
           1      0.745     0.854     0.796        48
           2      0.848     0.736     0.788        53

    accuracy                          0.853       150
   macro avg      0.858     0.857     0.855       150
weighted avg      0.858     0.853     0.853       150



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 10-Seed Test for GatedGAT Model

To evaluate the model's robustness and stability, we will run the training and evaluation process 10 times, each with a different random seed. We will then collect and report the mean and standard deviation of the key performance metrics.

In [ ]:
def run_experiment(seed, data, num_classes, num_features, device, class_weights):
    set_seed(seed)
    print(f"\n--- Running experiment with seed: {seed} ---")

    model_gated_gat = GatedGAT(num_features, hidden_dim=32, out_dim=num_classes, heads=8, dropout=0.5).to(device)
    history = train_gnn(model_gated_gat, data, epochs=300, lr=0.01, weight_decay=5e-4, patience=40,
                         verbose=False, class_weights=class_weights) # verbose=False for cleaner output during multiple runs

    eval_results = evaluate(model_gated_gat, data, data.test_mask)
    print(f"Macro-F1: {eval_results['macro_f1']:.4f}, High-Risk Recall: {eval_results['high_risk_recall']:.4f}")
    return eval_results


all_results = []
num_runs = 10
for i in range(num_runs):
    results = run_experiment(i, data, num_classes, num_features, device, class_weights)
    all_results.append(results)

# Aggregate results
macro_f1_scores = [res['macro_f1'] for res in all_results]
high_risk_recall_scores = [res['high_risk_recall'] for res in all_results]
mean_gate_values = [res['mean_gate'] for res in all_results]

print(f"\n--- Summary of {num_runs} Runs ---")
print(f"Mean Macro-F1: {np.mean(macro_f1_scores):.4f} +/- {np.std(macro_f1_scores):.4f}")
print(f"Mean High-Risk Recall: {np.mean(high_risk_recall_scores):.4f} +/- {np.std(high_risk_recall_scores):.4f}")
print(f"Mean Gate Value: {np.mean(mean_gate_values):.4f} +/- {np.std(mean_gate_values):.4f}")

#Print per-class F1 average/std if needed
print("\nPer-class F1 (Mean +/- Std Dev):")
class_names = ['High-Risk', 'Moderate-Risk', 'Low-Risk']
for class_name in class_names:
    per_class_f1_current = [res['per_class_f1'][class_name] for res in all_results]
    print(f"  {class_name}: {np.mean(per_class_f1_current):.4f} +/- {np.std(per_class_f1_current):.4f}")


--- Running experiment with seed: 0 ---
Macro-F1: 0.8537, High-Risk Recall: 0.9592

--- Running experiment with seed: 1 ---
Macro-F1: 0.8526, High-Risk Recall: 0.9796

--- Running experiment with seed: 2 ---
Macro-F1: 0.8268, High-Risk Recall: 0.9592

--- Running experiment with seed: 3 ---
Macro-F1: 0.8477, High-Risk Recall: 0.9592

--- Running experiment with seed: 4 ---
Macro-F1: 0.8741, High-Risk Recall: 0.9592

--- Running experiment with seed: 5 ---
Macro-F1: 0.8477, High-Risk Recall: 0.9592

--- Running experiment with seed: 6 ---
Macro-F1: 0.8411, High-Risk Recall: 0.9592

--- Running experiment with seed: 7 ---
Macro-F1: 0.8536, High-Risk Recall: 0.9592

--- Running experiment with seed: 8 ---
Macro-F1: 0.8394, High-Risk Recall: 0.9592

--- Running experiment with seed: 9 ---
Macro-F1: 0.8476, High-Risk Recall: 0.9592

--- Summary of 10 Runs ---
Mean Macro-F1: 0.8484 +/- 0.0116
Mean High-Risk Recall: 0.9612 +/- 0.0061
Mean Gate Value: 0.2068 +/- 0.0246

Per-class F1 (Mean +/-

In [ ]:
import torch, json, os
from datetime import datetime, timezone


SAVE_PATH = "/content/gated_gat_v1.pt"

checkpoint = {
    "state_dict": model_gated_gat.state_dict(),

    #architecture rebuild
    "arch": "GatedGAT",
    "config": {
        "in_dim":     num_features,
        "hidden_dim": 32,
        "out_dim":    num_classes,
        "heads":      8,
        "dropout":    0.5,
    },

    #Verify Output index
    "class_order": ["High", "Moderate", "Low"],

    # Provenance
    "graph_file":  "real_graph_relabeled.pt",
    "trained_at":  datetime.now(timezone.utc).isoformat(),
    "torch_version": torch.__version__,
    "test_macro_f1": float(eval_results["macro_f1"]),
    "test_high_risk_recall": float(eval_results["high_risk_recall"]),
}

torch.save(checkpoint, SAVE_PATH)
print("Saved:", SAVE_PATH)
print("Size:", round(os.path.getsize(SAVE_PATH) / 1e6, 2), "MB")

Saved: /content/gated_gat_v1.pt
Size: 0.2 MB


In [ ]:
from google.colab import files
files.download(SAVE_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
ckpt = torch.load("/content/gated_gat_v1.pt",
                  map_location="cpu", weights_only=False)

m = GatedGAT(**ckpt["config"])
m.load_state_dict(ckpt["state_dict"])
m.eval()

out, gate = m(data.x, data.edge_index)
pred = out[data.test_mask].argmax(dim=1).cpu().numpy()
true = data.y[data.test_mask].cpu().numpy()
from sklearn.metrics import f1_score
print("Reloaded macro-F1:", f1_score(true, pred, average="macro"))

Reloaded macro-F1: 0.8545290431559502


In [ ]:
from google.colab import files
files.download("/content/gated_gat_v1.pt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>